<h1><center><big>Semi-supervised learning : le pseudo-labeling</big> <br></center></h1>

<h3><center>Expérimentations sur CIFAR10</center></h3>
<hr>

Nous avons adapté le tutoriel pytorch pour faire du semi-supervisé :

https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html

Nous utiliserons CIFAR10, un jeu de données de petites images couleur de 32x32 pixels avec 10 classes différentes.

L'ensemble d'entraînement compte normalement 50 000 images et l'ensemble de test 10 000 images.

Ce jeu de données est entièrement annoté, nous allons donc supprimer artificiellement certaines étiquettes.

In [ ]:
#! nvidia-smi

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
#import torch.backends.cudnn as cudnn
import torch.backends.mps as mps

import matplotlib.pyplot as plt
import numpy as np

from tqdm.notebook import tqdm_notebook

Pour la reproductibilité :

In [ ]:
SEED=2023
torch.manual_seed(SEED)
torch.mps.manual_seed(SEED)
mps.benchmark = True #type: ignore
np.random.seed(SEED)
# random.seed(SEED)

In [ ]:
device = torch.device('mps:0' if torch.mps.is_available() else 'cpu')
# Assuming that we are on a CUDA machine, this should print a CUDA device:
print(device)

In [ ]:
rep_path= "/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_APPAUT/2026_TPIA3_2/"

# Chargement de CIFAR10

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 32

trainset = torchvision.datasets.CIFAR10(root=rep_path+'data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=0)

testset = torchvision.datasets.CIFAR10(root=rep_path+'data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=0)

classes = ('plane', 'car', 'bird', 'cat','deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
def imshow(images, labels, predicted_labels=None):
    # Using torchvision to make a grid of the images
    img = torchvision.utils.make_grid(images)

    img = img / 2 + 0.5     # unnormalize
    img = img.permute(1, 2, 0)

    # Plotting the grid
    fig, ax = plt.subplots(figsize=(6, 24))
    plt.imshow(img)

    if predicted_labels is not None:
        # labels prédits si elles existent
        ax.set_xlabel('Predicted labels', fontsize=14, labelpad=12)
        ax.set_xticks(torch.arange(len(images)) * 18 + 10)
        ax.set_xticklabels([classes[predicted_labels[j]]
                            for j in range(len(images))], fontsize=14)

    # labels ground truth
    gax = ax.secondary_xaxis('top')
    gax.set_xlabel('Ground truth', fontsize=14, labelpad=12)
    gax.set_xticks(torch.arange(len(images)) * 18 + 10)
    gax.set_xticklabels([classes[labels[j]]
                         for j in range(len(images))], fontsize=14)
    plt.show()

In [ ]:
# Images random du train
dataiter = iter(trainloader)
images, labels = next(dataiter)

In [ ]:
imshow(images[:4], labels[:4])

# Split artificiel du train

Divisons l'ensemble des données de train en :


* exemples avec label
* exemples sans label

Comme les données sont entièrement étiquetées, nous supprimons artificiellement certaines étiquettes et leur attribuons la valeur -1.



In [ ]:
# Nous ne gardons que 40% de l'ensemble des données étiquetées du train
# Vous pourrez essayer une autre valeur par la suite
# A priori que se passe-t-il si cette valeur est très faible ? Très élevée ?

proportion_labeled_elements = 0.4

# on shuffle les indices :
indices = torch.randperm(len(trainset))

n_labeled_indices = int(len(indices) * proportion_labeled_elements)
indices_labeled = sorted(indices[:n_labeled_indices])
indices_unlabeled = sorted(indices[n_labeled_indices:])

for index in indices_unlabeled:
    trainset.targets[index] = -1  # on met à -1 le label (valeur arbitraire, on la remplacera pour un label prédit par la suite)

dataset_train_labeled = torch.utils.data.Subset(trainset, indices_labeled) #type: ignore
dataset_train_unlabeled = torch.utils.data.Subset(trainset, indices_unlabeled) #type: ignore

In [ ]:
# Nous ne gardons que 40% de l'ensemble des données étiquetées du train
# Vous pourrez essayer une autre valeur par la suite
# A priori que se passe-t-il si cette valeur est très faible ? Très élevée ?

def dataset_labelization(trainset, proportion_labeled_elements):
    proportion_labeled_elements = proportion_labeled_elements
    # on shuffle les indices :
    indices = torch.randperm(len(trainset))

    n_labeled_indices = int(len(indices) * proportion_labeled_elements)
    indices_labeled = sorted(indices[:n_labeled_indices])
    indices_unlabeled = sorted(indices[n_labeled_indices:])

    for index in indices_unlabeled:
        trainset.targets[index] = -1  # on met à -1 le label (valeur arbitraire, on la remplacera pour un label prédit par la suite)

    dataset_train_labeled = torch.utils.data.Subset(trainset, indices_labeled) #type: ignore
    dataset_train_unlabeled = torch.utils.data.Subset(trainset, indices_unlabeled) #type: ignore
    print(f'len dataset_train_labeled {len(dataset_train_labeled)}, len dataset_train_unlabeled : {len(dataset_train_unlabeled)}')
    return dataset_train_labeled, dataset_train_unlabeled

In [ ]:
def label_dataset(loader, model,threshold:float=0):
    """
    Retourne les prédictions sur un subset donné par loader
    """
    with torch.no_grad():
        model.eval()
        all_labels = []
        n_correct=0
        n_total=0
        for i, data in tqdm_notebook(enumerate(loader), total=len(loader)):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)           
            # forward + backward + optimize
            outputs = model(inputs)
            predicted = torch.argmax(outputs, dim=1)
            predicted = predicted.detach().cpu()
            if threshold != 0 : 
                probs = torch.softmax(outputs, dim=1)
                mask = probs[:,predicted] >= threshold
                mask = mask.detach().cpu()
                selected = predicted[mask[i]] 
                all_labels.extend(selected) 
            else:     
                all_labels.extend(predicted)
        return all_labels

In [ ]:
def train_loaderization(net, batch_size, trainset,dataset_train_labeled,dataset_train_unlabeled,threshold): 
    train_loader_labeled = torch.utils.data.DataLoader(dataset_train_labeled, batch_size=batch_size, shuffle=True)
    # Pas de shuffle sur la data loader des unlabeled, sinon on ne pourra pas les remplacer facilement par des labels prédits
    train_loader_unlabeled = torch.utils.data.DataLoader(dataset_train_unlabeled, batch_size=batch_size)
    labels = label_dataset(train_loader_unlabeled, net,threshold)
    for k, index in enumerate(labels):
        trainset.targets[index] = labels[k]
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)    
    return train_loader  

In [ ]:
train_loader_labeled = torch.utils.data.DataLoader(dataset_train_labeled, batch_size=batch_size, shuffle=True)
# Pas de shuffle sur la data loader des unlabeled, sinon on ne pourra pas les remplacer facilement par des labels prédits
train_loader_unlabeled = torch.utils.data.DataLoader(dataset_train_unlabeled, batch_size=batch_size)

# Définition d'un petit CNN

Vous pourrez par la suite essayer un modèle standard, comme un ResNet, pré-entraîné sur ImageNet ou non.

Voir si le pseudo-labeling que l'on fait ici avec un tout petit modèle marche aussi avec un plus gros modèle comme ResNet.


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
def train(testloader, trainloader, model, opt, crit, n_epoch=2, loss_every=500):
    """
    Entraînement d'un modèle et plot des courbes de loss et accuracy
    """
    model.train()
    losses = []
    acc = []
    for epoch in range(n_epoch):  # loop over the dataset multiple times
        #print(f"Epoch {epoch}.")

        running_loss = []
        running_acc = []
        #print("Train.")
        #for i, data in tqdm_notebook(enumerate(trainloader), total=len(trainloader)):
        for i, data in enumerate(trainloader):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            # Mettre à zero les gradients des poids du modèle
            opt.zero_grad()

            # forward + backward + optimize
            outputs = model(inputs)
            loss = crit(outputs, labels)
            loss.backward()
            opt.step()

            predicted = torch.argmax(outputs, dim=1)

            running_loss.append(loss.item())
            running_acc.append((predicted == labels).sum().item() / labels.size(0))

            # calculer une moyenne
            if i % loss_every:
                losses.append(np.mean(running_loss))
                acc.append(np.mean(running_acc))

                running_loss = []
                running_acc = []

        test_acc = accuracy(testloader, model)
        print(f"Train Epoch {epoch} Test accuracy: {test_acc:.3f}")

    fig, axes = plt.subplots(1, 2)
    axes[0].plot(losses)
    axes[1].plot(acc)

    axes[0].set_ylabel("Train loss")
    axes[1].set_ylabel("Train acc")
    plt.show()
    print('Apprentissage terminé')


def accuracy(loader, model):
    """
    Args:
        loader: data loader sur lequel on veut calculer une accuracy
        model
    Returns:
        Accuracy
    """
    with torch.no_grad():
        model.eval()  # remove potential dropout, ...
        n_correct = 0
        n_total = 0
 #       for i, data in tqdm_notebook(enumerate(loader), total=len(loader)):
        for i, data in enumerate(loader):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            # forward + backward + optimize
            outputs = model(inputs)
            predicted = torch.argmax(outputs, dim=1)
            n_correct += (predicted == labels).sum()
            n_total += labels.size(0)
        return n_correct / n_total


def validate(loader, model):
    """
    Plot des predictions faites avec model, affiche l'accuracy
    """
    dataiter = iter(loader)
    # Get one batch of data
    images, labels = next(dataiter)
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)
    predictions = torch.argmax(outputs, dim=1)

    accuracy_model = accuracy(loader, model)
    # print images
    print(f'Accuracy: {accuracy_model:.3f}')
    imshow(images[:4].detach().cpu(), labels[:4], predicted_labels=predictions[:4])

### `Train avec labels (les 40 % du train supervisé)`

__`1. Préparation des données`__
 - Séparer un petit jeu de données étiquetées L et un grand jeu non étiqueté U.
 - Nettoyer, normaliser, faire du split train/val/test sur la partie étiquetée comme en supervisé classique

In [ ]:
train_loader_labeled = torch.utils.data.DataLoader(dataset_train_labeled, batch_size=batch_size, shuffle=True)
# Pas de shuffle sur la data loader des unlabeled, sinon on ne pourra pas les remplacer facilement par des labels prédits
train_loader_unlabeled = torch.utils.data.DataLoader(dataset_train_unlabeled, batch_size=batch_size)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,shuffle=False, num_workers=0)

__`2. Entraînement supervisé initial`__
 - Entraîner un modèle de base uniquement sur L (apprentissage supervisé standard).
 - Vérifier ses performances sur un jeu de validation pour disposer d’un point de départ fiable.

In [ ]:
net = Net().to(device)
# tu peux mettre -1 dans ton tenseur de labels pour toutes les positions à “masquer” 
# CrossEntropyLoss(ignore_index=-1) se chargera de ne pas les compter dans la loss ni dans le backprop
criterion = nn.CrossEntropyLoss() # 
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
train(testloader, train_loader_labeled, net, optimizer, criterion, n_epoch=5)

In [ ]:
validate(testloader, net)

#### `Utiliser le modèle sur le train unlabeled et utiliser les labels prédits`
__`3. Génération de pseudo-labels`__
 - Appliquer le modèle sur U pour prédire des labels pour les données non étiquetées (pseudo-labels).
 - Ne garder que les prédictions jugées fiables (par exemple probabilité > seuil), afin de limiter le bruit.

In [ ]:
def label_dataset(loader, model,threshold:float=0):
    """
    Retourne les prédictions sur un subset donné par loader
    """
    with torch.no_grad():
        model.eval()
        all_labels = []
        n_correct=0
        n_total=0
        for i, data in tqdm_notebook(enumerate(loader), total=len(loader)):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)           
            # forward + backward + optimize
            outputs = model(inputs)
            predicted = torch.argmax(outputs, dim=1)
            predicted = predicted.detach().cpu()
            if threshold != 0 : 
                probs = torch.softmax(outputs, dim=1)
                mask = probs[:,predicted] >= threshold
                mask = mask.detach().cpu()
                selected = predicted[mask[i]] 
                all_labels.extend(selected) 
            else:     
                all_labels.extend(predicted)
        return all_labels

#### `Train du modèle avec la partie labeled + unlabeled avec les "pseudo-labels"`
Remplacer les labels dans trainset par les labels prédits vs"pseudo-labels" pour le dataset unlabeled
__`4. Ré-entraînement avec L ∪ U pseudo-labellé`__
 - Ajouter les exemples de U avec pseudo-labels au jeu d’entraînement et ré-entraîner le modèle.
- Utiliser éventuellement une pondération plus faible pour les pseudo-labels que pour les labels “vrais”

In [ ]:
for data in train_loader_unlabeled:
    print(data)

In [ ]:
labels = label_dataset(train_loader_unlabeled, net)

In [ ]:

for k, index in enumerate(labels):
    trainset.targets[index] = labels[k]   

In [ ]:

train_loader_labeled = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)    

In [ ]:
net2 = Net().to(device)
optimizer = torch.optim.AdamW(net2.parameters(), lr=1e-3, weight_decay=1e-2)
train(testloader, trainloader, net2, optimizer, criterion, n_epoch=5)
validate(testloader, net2)

In [ ]:
validate(testloader, net2)

L'accuracy doit être meilleure, le pseudo-labeling aide !

#### `Comparaison avec un modèle entraîné sur 100 % du train avec les vraies labels`

In [ ]:
net3 = Net().to(device)
optimizer = optim.SGD(net3.parameters(), lr=0.001, momentum=0.9)

In [ ]:
train(testloader, trainloader, net3, optimizer, criterion, n_epoch=5)

On obtient environ 10 points de mieux quand même que précedemment !

#### `Faire varier les proportions de données avec et sans labels`

In [ ]:
def train_loaderization(net, batch_size, trainset,dataset_train_labeled,dataset_train_unlabeled,threshold): 
    train_loader_labeled = torch.utils.data.DataLoader(dataset_train_labeled, batch_size=batch_size, shuffle=True)
    # Pas de shuffle sur la data loader des unlabeled, sinon on ne pourra pas les remplacer facilement par des labels prédits
    train_loader_unlabeled = torch.utils.data.DataLoader(dataset_train_unlabeled, batch_size=batch_size)
    labels = label_dataset(train_loader_unlabeled, net,threshold)
    for k, index in enumerate(labels):
        trainset.targets[index] = labels[k]
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)    
    return train_loader       

##### `100% labeled only`

In [ ]:
net3 = Net().to(device)
optimizer = optim.SGD(net3.parameters(), lr=0.001, momentum=0.9)
train(testloader, trainloader, net3, optimizer, criterion, n_epoch=5)

##### `40% labeled only`

In [ ]:
from torch.utils.data import random_split, DataLoader

def ration_dataset(dataset_train_labeled,ratio=0.4):
    n_total = len(dataset_train_labeled)
    n_sub = int(ratio * n_total)
    n_rest = n_total - n_sub
    train_subset, _ = random_split(dataset_train_labeled, [n_sub, n_rest])
    return train_subset

train_subset = ration_dataset(dataset_train_labeled,0.4)
trainloader = DataLoader(train_subset,batch_size=batch_size,shuffle=True,num_workers=0,
)

net3 = Net().to(device)
optimizer = optim.SGD(net3.parameters(), lr=0.001, momentum=0.9)
train(testloader, trainloader, net3, optimizer, criterion, n_epoch=5)

##### `10% labeled only`

In [ ]:

def ration_dataset(dataset_train_labeled,ratio=0.1):
    n_total = len(dataset_train_labeled)
    n_sub = int(ratio * n_total)
    n_rest = n_total - n_sub
    train_subset, _ = random_split(dataset_train_labeled, [n_sub, n_rest])
    return train_subset

train_subset = ration_dataset(dataset_train_labeled,0.4)
trainloader = DataLoader(train_subset,batch_size=batch_size,shuffle=True,num_workers=0,
)
net3 = Net().to(device)
optimizer = optim.SGD(net3.parameters(), lr=0.001, momentum=0.9)
train(testloader, trainloader, net3, optimizer, criterion, n_epoch=5)

##### `20% labeled and 80% unlabeled`

In [ ]:
dataset_train_labeled, dataset_train_unlabeled = dataset_labelization(trainset, 0.2)
print(f'len(dataset_train_labeled) : {len(dataset_train_labeled)} len(dataset_train_unlabeled) : {len(dataset_train_unlabeled)}')
train_loader = train_loaderization(net3, batch_size, trainset,dataset_train_labeled,dataset_train_unlabeled,0)
net3 = Net().to(device)
optimizer = optim.SGD(net3.parameters(), lr=0.001, momentum=0.9)
train(testloader, trainloader, net3, optimizer, criterion, n_epoch=5)

##### `10% labeled and 90% unlabeled`

In [ ]:
dataset_train_labeled, dataset_train_unlabeled = dataset_labelization(trainset, 0.1)
print(f'len(dataset_train_labeled) : {len(dataset_train_labeled)} len(dataset_train_unlabeled) : {len(dataset_train_unlabeled)}')
train_loader = train_loaderization(net3, batch_size, trainset,dataset_train_labeled,dataset_train_unlabeled,0)
net3 = Net().to(device)
optimizer = optim.SGD(net3.parameters(), lr=0.001, momentum=0.9)
train(testloader, trainloader, net3, optimizer, criterion, n_epoch=5)

#### `40% labeled and 60% unlabeled data`

In [ ]:
dataset_train_labeled, dataset_train_unlabeled = dataset_labelization(trainset, 0.4)
print(f'len(dataset_train_labeled) : {len(dataset_train_labeled)} len(dataset_train_unlabeled) : {len(dataset_train_unlabeled)}')
train_loader = train_loaderization(net3, batch_size, trainset,dataset_train_labeled,dataset_train_unlabeled,0)
net3 = Net().to(device)
optimizer = optim.SGD(net3.parameters(), lr=0.001, momentum=0.9)
train(testloader, trainloader, net3, optimizer, criterion, n_epoch=5)

In [ ]:
def label_dataset(loader, model,threshold:float=0):
    """
    Retourne les prédictions sur un subset donné par loader
    """
    with torch.no_grad():
        model.eval()
        all_labels = []
        n_correct=0
        n_total=0
        for i, data in tqdm_notebook(enumerate(loader), total=len(loader)):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            
            # forward + backward + optimize
            outputs = model(inputs)
            predicted = torch.argmax(outputs, dim=1) 
            predicted = predicted.detach().cpu()
            if threshold != 0 :
                probs = torch.softmax(outputs, dim=1)
                mask = probs[:,predicted] >= threshold
                mask = mask.detach().cpu()
                selected = predicted[mask[i]] 
                all_labels.extend(selected) 
                print("par ici") 
                return all_labels      
            else: 
                print("par la")    
                all_labels.extend(predicted)
                return all_labels

In [ ]:
dataset_train_labeled, dataset_train_unlabeled = dataset_labelization(trainset, 0.4)
print(f'len(dataset_train_labeled) : {len(dataset_train_labeled)} len(dataset_train_unlabeled) : {len(dataset_train_unlabeled)}')
train_loader = train_loaderization(net3, batch_size, trainset,dataset_train_labeled,dataset_train_unlabeled,0)
net3 = Net().to(device)
optimizer = optim.SGD(net3.parameters(), lr=0.001, momentum=0.9)
train(testloader, trainloader, net3, optimizer, criterion, n_epoch=5)

Quelques idées d'expérimentations :



*   Baisser à 20% puis 10% la proportion de données avec labels
*   Revenir à 40 % mais après avoir fait des prédictions sur les 60 % restant, 
*   sélectionner parmi ces 60 % uniquement les exemples dont la confiance (la probabilité associée à la classe prédite par le réseau) est supérieure à un seuil qui vous choississez, par exemple 0.9. 


Pour faciliter l'implantation de ce seuillage, vous pouvez mettre les labels des exemples non-sélectionnés à **-1** et utiliser l'option **ignore_index=-1** de **CrossEntropyLoss** L'inconvénient de cette méthode plutôt que de recréer un nouveau subset de train, cependant, est de faire les calculs du réseau même sur les exemples non-sélectionnés, qui ne servent pas à l'entraînement.

*   Tester avec un ResNet pré-entraîné ou non

